In [1]:
import pandas as pd, numpy as np, glob

path = glob.glob('/kaggle/input/**/creditcard.csv', recursive=True)[0]
df = pd.read_csv(path)

print(df.shape)
print(df['Class'].value_counts())
print("Fraud %:", round(df['Class'].mean() * 100, 3))

(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64
Fraud %: 0.173


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df = df.drop(['Time', 'Amount'], axis=1)

X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

(227845, 29) (56962, 29)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_score = lr.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, lr_pred))
print(classification_report(y_test, lr_pred, digits=3))

[[55423  1441]
 [    8    90]]
              precision    recall  f1-score   support

           0      1.000     0.975     0.987     56864
           1      0.059     0.918     0.110        98

    accuracy                          0.975     56962
   macro avg      0.529     0.947     0.549     56962
weighted avg      0.998     0.975     0.986     56962



In [4]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=float(y_train.mean()),
                      random_state=42, n_jobs=-1)
iso.fit(X_train)
iso_pred = (iso.predict(X_test) == -1).astype(int)
iso_score = -iso.score_samples(X_test)

print(confusion_matrix(y_test, iso_pred))
print(classification_report(y_test, iso_pred, digits=3))

[[56796    68]
 [   66    32]]
              precision    recall  f1-score   support

           0      0.999     0.999     0.999     56864
           1      0.320     0.327     0.323        98

    accuracy                          0.998     56962
   macro avg      0.659     0.663     0.661     56962
weighted avg      0.998     0.998     0.998     56962



In [5]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

rows = []
for name, pred, score in [("Logistic Regression", lr_pred, lr_score),
                          ("Isolation Forest", iso_pred, iso_score)]:
    rows.append({
        "Model": name,
        "Precision": round(precision_score(y_test, pred), 3),
        "Recall": round(recall_score(y_test, pred), 3),
        "F1": round(f1_score(y_test, pred), 3),
        "PR-AUC": round(average_precision_score(y_test, score), 3),
    })
print(pd.DataFrame(rows))

                 Model  Precision  Recall     F1  PR-AUC
0  Logistic Regression      0.059   0.918  0.110   0.716
1     Isolation Forest      0.320   0.327  0.323   0.171
